# Exercise: Fix the RAG Pipeline Issues

## Objective
Debug and fix a broken RAG (Retrieval Augmented Generation) pipeline by identifying and resolving 5 critical issues that prevent it from working correctly.

## What You'll Do
The first code cell contains a RAG pipeline with **5 marked issues** (labeled as ISSUE #1 through ISSUE #5). Your task is to identify what's wrong with each part and fix it.

### The 5 Issues to Fix

| Issue | Component | Problem |
|-------|-----------|---------|
| **ISSUE #1** | Vector Store | Initialization or configuration problem |
| **ISSUE #2** | Retrieval | Wrong parameter value (k is too high) |
| **ISSUE #3** | Ranking | No ranking logic implemented |
| **ISSUE #4** | Context Builder | Only uses first document (loses context) |
| **ISSUE #5** | LLM Call | Context retrieved but not actually used in the message |

## How This Exercise Works

1. **First cell**: The broken RAG pipeline with all 5 issues marked with ❌
2. **Following cells**: Individual solution cells showing the correct implementation for each issue
3. **Your task**: Review each issue, understand what's wrong, and see how it should be fixed

## What You'll Learn

- How to identify bugs in a RAG system
- The importance of:
  - **Proper retrieval parameters** (k = number of results)
  - **Ranking/scoring** to prioritize relevant documents
  - **Full context building** (using all relevant docs, not just one)
  - **Actually using the context** in your LLM prompt
  - **Proper prompt engineering** (including both context and query)

## Key Concepts

- **Retrieval**: Fetching relevant documents from a vector store
- **Ranking**: Scoring and sorting documents by relevance
- **Context Building**: Combining multiple documents into a coherent context string
- **Prompt Engineering**: Including retrieved context in the LLM input to improve answer quality

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings


# -------------------------
# Sample Documents
# -------------------------
documents = [
    "Subscription pricing works best for SaaS.",
    "Freemium model failed in enterprise.",
    "Tiered pricing increased revenue.",
    "Customers like flexible pricing."
]


# -------------------------
# Vector Store (ISSUE #1)
# -------------------------
embeddings = OpenAIEmbeddings()

vector_store = Chroma.from_texts(
    documents,
    embedding=embeddings
)


# -------------------------
# Retrieval (ISSUE #2)
# -------------------------
def retrieve(query):

    # ❌ wrong k (too high)
    return vector_store.similarity_search(query, k=10)


# -------------------------
# Ranking (ISSUE #3)
# -------------------------
def rank(docs):

    # ❌ no ranking logic
    return docs


# -------------------------
# Context Builder (ISSUE #4)
# -------------------------
def build_context(docs):

    # ❌ only takes first doc (loses context)
    return docs[0].page_content


# -------------------------
# LLM Call (ISSUE #5)
# -------------------------
def generate_answer(query):

    docs = retrieve(query)

    docs = rank(docs)

    context = build_context(docs)

    llm = ChatOpenAI(model="gpt-4o", temperature=0)

    messages = [
        SystemMessage(content="You are helpful."),
        HumanMessage(content=f"{query}")  # ❌ context not used
    ]

    return llm.invoke(messages).content


# -------------------------
# Run
# -------------------------
print(generate_answer("What pricing strategy should we use?"))

In [ ]:
def retrieve(query):
    return vector_store.similarity_search(query, k=3)

In [ ]:
def rank(docs, query):

    scored = []

    for doc in docs:
        score = 0
        for word in query.lower().split():
            if word in doc.page_content.lower():
                score += 1

        scored.append((doc, score))

    scored.sort(key=lambda x: x[1], reverse=True)

    return [d for d, _ in scored]

In [ ]:
def build_context(docs):

    return "\n\n".join([doc.page_content for doc in docs])

In [ ]:
messages = [
    SystemMessage(content="You are a product strategist."),
    HumanMessage(content=f"""
Context:
{context}

Question:
{query}
""")
]

In [ ]:
# cache results
cache = {}